In [25]:
# imports

from langchain_core.documents import Document 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from pathlib import Path

In [11]:
docs_by_hand = [
    Document(page_content="Python is a programming language", metadata={"source": "intro.txt", "level": "beginner"}),
    Document(page_content="Embeddings turn text into vectors", metadata={"source": "intro1.txt", "level": "beginner"})
]

print(docs_by_hand[0])
print([d.page_content for d in docs_by_hand])

page_content='Python is a programming language' metadata={'source': 'intro.txt', 'level': 'beginner'}
['Python is a programming language', 'Embeddings turn text into vectors']


In [12]:
PDFS = Path("pdfs")

pdf_path = sorted(PDFS.glob("*.pdf"))[0] # sort the pdfs

print(pdf_path)

pdfs/brightloop_handbook.pdf


In [13]:
loader = PyPDFLoader(str(pdf_path))


In [16]:
pages = loader.load()

print(type(pages))
print(len(pages))
print(pages[0])

<class 'list'>
6
page_content='Welcome to Brightloop
Brightloop is a 74-person company in Leeds that makes route-planning software for bike couriers.
We were founded in 2019. Core hours are 10:30 to 15:30 UK time, and the whole company meets
for a stand-up every Tuesday at 09:45. Outside core hours you are free to arrange your day, as
long as your team knows when to reach you. We write down decisions in the Lighthouse channel so
that nobody has to ask twice. New starters get a buddy for their first three months, and the buddy
books a coffee chat every Friday. Our values are short: be kind, be clear, and ship small. Visitors
must sign in at the front desk and wear a badge. The office is closed on the last Friday of
December, and nobody is expected to read messages that day. Every team keeps a public
changelog so that others can follow its progress.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-11T13:06:30+01:00', 'aut

In [17]:
for i, page in enumerate(pages):
    print(f"Page {i+1}:")
    print(page.page_content)
    print("\n")



Page 1:
Welcome to Brightloop
Brightloop is a 74-person company in Leeds that makes route-planning software for bike couriers.
We were founded in 2019. Core hours are 10:30 to 15:30 UK time, and the whole company meets
for a stand-up every Tuesday at 09:45. Outside core hours you are free to arrange your day, as
long as your team knows when to reach you. We write down decisions in the Lighthouse channel so
that nobody has to ask twice. New starters get a buddy for their first three months, and the buddy
books a coffee chat every Friday. Our values are short: be kind, be clear, and ship small. Visitors
must sign in at the front desk and wear a badge. The office is closed on the last Friday of
December, and nobody is expected to read messages that day. Every team keeps a public
changelog so that others can follow its progress.


Page 2:
Parental leave and time off
Brightloop gives every parent 17 weeks of fully paid parental leave, including adoption and
surrogacy. After that, parents ca

In [18]:
for i, page in enumerate(loader.lazy_load()):
    print(page.metadata, "---", len(page.page_content))
    if i == 1: break

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-11T13:06:30+01:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-11T13:06:30+01:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'pdfs/brightloop_handbook.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'} --- 828
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-11T13:06:30+01:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-11T13:06:30+01:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'pdfs/brightloop_handbook.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'} --- 738


In [20]:
toy = "Brightloop gives 17 weeks of paid parental leave.\n\nMeals are reimbursed up to 38 pounds per day when you travel."


In [21]:
CharacterTextSplitter(separator="\n\n", chunk_size=25, chunk_overlap=0).split_text(toy)

Created a chunk of size 49, which is longer than the specified 25


['Brightloop gives 17 weeks of paid parental leave.',
 'Meals are reimbursed up to 38 pounds per day when you travel.']

In [23]:
RecursiveCharacterTextSplitter(chunk_size=25, chunk_overlap=0).split_text(toy)

['Brightloop gives 17 weeks',
 'of paid parental leave.',
 'Meals are reimbursed up',
 'to 38 pounds per day',
 'when you travel.']

### https://docs.langchain.com/oss/python/integrations/splitters/split_by_token#tiktoken

### https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter/from_language

# FAISS

In [26]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

/Users/sanket/Developer/Codebases/AiEngineeringConcepts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 38119.94it/s]


In [27]:
FACTS = [
    "Zorvia Coffee annual subscriptions can be refunded within 14 days of purchase.",
    "Monthly subscriptions can be cancelled at any time, but the current month is not refunded.",
    "Orders of 5 bags or more ship free.",
    "Zorvia roasts every Tuesday and Friday in Lisbon.",
    "Customer support is open 9:00 to 17:00 UTC, Monday to Friday.",
    "The Ember blend is a dark roast with notes of cocoa and orange peel.",
    "The Drift blend is a light roast from Ethiopia with notes of jasmine and lemon.",
    "Zorvia gift cards never expire.",
    "Beans are freshest for 30 days after the roast date if kept in a sealed bag.",
    "Wholesale accounts require a minimum order of 20 kg per month.",
    "In the referral program, both the referrer and the friend get one free bag.",
    "Orders placed before 12:00 UTC ship the same day.",
]

In [28]:
def to_documents(facts: list[str]) -> list[Document]:
    return [Document(page_content=text, metadata={"id": i}) for i, text in enumerate(facts)]

In [30]:
D = to_documents(FACTS)

In [33]:
vectorstore = FAISS.from_documents(D, embedding)

In [42]:
query = "What is the refund policy for annual subscriptions?"
results = vectorstore.similarity_search_with_score(query, k=2)


for doc, score in results:
    print(doc.page_content, score)



Monthly subscriptions can be cancelled at any time, but the current month is not refunded. 0.63275766
Zorvia Coffee annual subscriptions can be refunded within 14 days of purchase. 1.0218582


In [44]:
FAISS_INDEX_PATH = "./faiss_index"
vectorstore.save_local(FAISS_INDEX_PATH)